# EE-2.2: Active, reactive and complex power

## Objectives

At the end of this notebook you should be able to:

- explain why instantaneous AC power oscillates, sometimes flows **backwards**, and what its average means
- define **active power P** (W), **reactive power Q** (var) and **apparent power |S|** (VA), and compute them with **S = V · I\*** (the conjugate matters)
- use the **power factor**, and size a capacitor for **power factor correction**
- show that complex power is conserved in a network
- derive the **power transfer equation** P = V₁V₂ sin δ / X between two buses, and quantify the error of its linear approximation P ≈ δ / X — the **DC power flow** used by PyPSA and by European market coupling

In EE-2.1 current and voltage became phasors that can be shifted against each other. This lesson answers the obvious next question: if they are shifted, how much power actually flows? The answer splits power into two kinds — and explains why grids care about something called reactive power, which does no work at all.

In [1]:
import numpy as np
import pandas as pd

f = 50.0
omega = 2 * np.pi * f

### Instantaneous power

Power at any instant is p(t) = v(t) · i(t). Take the RL circuit from EE-2.1 (230 V, 10 Ω, 50 mH), where the current lags the voltage by 57.5°, and compute p(t) over one period:

In [2]:
R, L = 10.0, 0.05
Z = R + 1j * omega * L
V = 230 + 0j
I = V / Z

t = np.linspace(0, 1 / f, 100_000, endpoint=False)
v = np.sqrt(2) * abs(V) * np.cos(omega * t)
i = np.sqrt(2) * abs(I) * np.cos(omega * t + np.angle(I))
p = v * i
print(f"p(t) ranges from {p.min():,.0f} W to {p.max():,.0f} W; negative {100 * (p < 0).mean():.0f} % of the time")
print(f"average power over one period: {p.mean():,.1f} W")

p(t) ranges from -1,315 W to 4,367 W; negative 32 % of the time
average power over one period: 1,525.6 W


Power oscillates at **twice** the grid frequency, and for about a third of each period it is **negative**: energy stored in the inductor's magnetic field flows back to the source. Only the **average**, 1,526 W, does useful work (heat in the resistor, torque in a motor). The back-and-forth exchange does no work, but it still needs current — and current causes losses and occupies line capacity.

### Complex power: S = V · I\*

Both parts are captured by one complex number, the **complex power**

**S = V · I\* = P + jQ**

where I\* is the complex **conjugate** of the current phasor:

| Quantity | Symbol | Unit | Meaning |
|---|---|---|---|
| active power | P = Re S = \|V\|\|I\| cos φ | W | average power, does work |
| reactive power | Q = Im S = \|V\|\|I\| sin φ | var | amplitude of the back-and-forth exchange |
| apparent power | \|S\| = \|V\|\|I\| | VA | what the equipment must carry (sizes cables, transformers) |
| power factor | cos φ = P / \|S\| | — | share of the current that does work |

φ is the angle between voltage and current. By convention, inductive loads **consume** reactive power (Q > 0) and capacitors **produce** it (Q < 0 when counted as consumption). Check every definition against the physics:

In [3]:
S = V * np.conj(I)
P, Q = S.real, S.imag
assert np.isclose(P, p.mean()), "P is the average of instantaneous power"
assert np.isclose(P, abs(I) ** 2 * R), "all active power is turned into heat in R"
assert np.isclose(Q, abs(I) ** 2 * omega * L), "all reactive power is exchanged with the inductor"
print(f"S = {P:,.1f} W + j{Q:,.1f} var | |S| = {abs(S):,.1f} VA | power factor {P / abs(S):.3f}")
print(f"without the conjugate, V*I = {V * I:,.1f}  <- the sign of Q is wrong")

S = 1,525.6 W + j2,396.5 var | |S| = 2,840.9 VA | power factor 0.537
without the conjugate, V*I = 1,525.6-2,396.5j  <- the sign of Q is wrong


Every number checks out: P equals the average power and the I²R heat, Q equals I²X in the inductor. Forgetting the conjugate flips the sign of Q — a load that consumes reactive power would appear to produce it. This is one of the most common bugs in hand-written power flow code; the check above catches it.

A power factor of 0.54 means the circuit draws almost twice the current that its active power alone would need.

### Power factor correction

A capacitor in parallel with the load produces reactive power locally, so the source no longer has to supply it through the line. To bring the power factor to a target value, the capacitor must supply Q_C = Q − P · tan(arccos(pf_target)), and a capacitor at voltage V supplies |V|² ωC, so

**C = Q_C / (|V|² ω)**

In [4]:
rows = []
for pf_target in [0.95, 1.00]:
    Q_C = Q - P * np.tan(np.arccos(pf_target))
    C = Q_C / (abs(V) ** 2 * omega)
    Z_total = 1 / (1 / Z + 1j * omega * C)              # load in parallel with the capacitor
    I_new = V / Z_total
    S_new = V * np.conj(I_new)
    assert np.isclose(S_new.real, P), "the load's active power is unchanged"
    rows.append({"target_pf": pf_target, "C_uF": C * 1e6, "source_current_A": abs(I_new),
                 "current_reduction_pct": 100 * (1 - abs(I_new) / abs(I)),
                 "line_loss_reduction_pct": 100 * (1 - (abs(I_new) / abs(I)) ** 2)})
pd.DataFrame(rows).round({"target_pf": 2, "C_uF": 1, "source_current_A": 2,
                          "current_reduction_pct": 1, "line_loss_reduction_pct": 1})

   target_pf   C_uF  source_current_A  current_reduction_pct  \
0       0.95  114.0              6.98                   43.5   
1       1.00  144.2              6.63                   46.3   

   line_loss_reduction_pct  
0                     68.0  
1                     71.2  

A 114 µF capacitor raises the power factor from 0.54 to 0.95 and cuts the current from the source by 43 % — and losses in the supply line, which scale with I², by 68 % — while the load gets exactly the same active power. Going all the way to 1.00 needs 26 % more capacitance for only 5 % less current: diminishing returns, which is why 0.95 is a common target. This is why grid codes require minimum power factors and why industrial customers are charged for reactive energy.

### Complex power is conserved

Kirchhoff's laws imply that complex power balances in every network: the complex power delivered by the sources equals the sum of S = |I|²·Z absorbed by all elements — active **and** reactive separately. Check it on EE-2.1's two-node example (a 230 V source, a line of 0.5 + j2 Ω, a load of 20 + j10 Ω):

In [5]:
Z_line, Z_load = 0.5 + 2j, 20 + 10j
I_ser = 230 / (Z_line + Z_load)
S_source = 230 * np.conj(I_ser)
S_line, S_load = abs(I_ser) ** 2 * Z_line, abs(I_ser) ** 2 * Z_load
assert np.isclose(S_source, S_line + S_load), "complex power balance"
print(f"source {S_source:,.1f} VA = line {S_line:,.1f} VA + load {S_load:,.1f} VA")
print(f"the line consumes {S_line.real:,.1f} W (losses) and {S_line.imag:,.1f} var (its own reactance)")

source 1,921.9+1,125.0j VA = line 46.9+187.5j VA + load 1,875.1+937.5j VA
the line consumes 46.9 W (losses) and 187.5 var (its own reactance)


The line itself **consumes** reactive power — four times more than its active losses, because its reactance is four times its resistance. Long, heavily loaded lines absorb large amounts of reactive power; that is why voltage support (reactive power from generators, capacitors, or inverters) is needed along the grid.

### Power transfer between two buses

Now the equation behind all of power flow. Two buses with voltages V₁ = |V₁|∠δ and V₂ = |V₂|∠0 are connected by a line with reactance X (resistance neglected, as for transmission lines). The power sent from bus 1 is S₁₂ = V₁ · ((V₁ − V₂) / jX)\*. Working it out gives

**P₁₂ = |V₁||V₂| sin δ / X**  and  **Q₁₂ = (|V₁|² − |V₁||V₂| cos δ) / X**

Two famous consequences: **active power flows from the leading to the lagging voltage angle**, and it is limited — at most |V₁||V₂| / X, reached at δ = 90°. Check the formula against the direct complex calculation for a 220 kV line with X = 40 Ω (voltages in V, powers in MW; EE-3 shows that with line-to-line voltages this same formula gives the total three-phase power):

In [6]:
V_mag, X = 220e3, 40.0
rows = []
for delta_deg in [5, 10, 20, 30, 60, 90]:
    d = np.radians(delta_deg)
    V1, V2 = V_mag * np.exp(1j * d), V_mag + 0j
    I12 = (V1 - V2) / (1j * X)
    S12, S21 = V1 * np.conj(I12), V2 * np.conj(-I12)
    P_formula = V_mag * V_mag * np.sin(d) / X
    assert np.isclose(S12.real, P_formula), "P = V1 V2 sin(delta) / X"
    assert np.isclose(S12.real, -S21.real), "a lossless line: what is sent arrives"
    rows.append({"delta_deg": delta_deg, "P12_MW": S12.real / 1e6, "Q12_Mvar": S12.imag / 1e6,
                 "P_DC_approx_MW": V_mag**2 * d / X / 1e6})
transfer = pd.DataFrame(rows)
transfer["DC_error_pct"] = 100 * (transfer.P_DC_approx_MW - transfer.P12_MW) / transfer.P12_MW
transfer.round(2)

   delta_deg   P12_MW  Q12_Mvar  P_DC_approx_MW  DC_error_pct
0          5   105.46      4.60          105.59          0.13
1         10   210.11     18.38          211.18          0.51
2         20   413.84     72.97          422.37          2.06
3         30   605.00    162.11          633.55          4.72
4         60  1047.89    605.00         1267.11         20.92
5         90  1210.00   1210.00         1900.66         57.08

The formula matches the complex calculation at every angle, and the lossless line delivers exactly what it receives. Read the columns:

- **P grows almost linearly with δ at small angles** and saturates at 1,210 MW at 90° — the **steady-state stability limit**. Beyond it, more angle means *less* power; operators keep transmission angles far below it.
- **The DC approximation**: with |V| ≈ nominal and small δ, sin δ ≈ δ, so **P ≈ V²·δ / X** — linear in the angle. Its error is 0.13 % at 5°, 0.5 % at 10°, 2 % at 20°, but 21 % at 60°. Normal transmission angles are mostly in the small range, which is why the linear **DC power flow** — the basis of PTDFs (EE-1.2), PyPSA's network optimisation and European flow-based market coupling — works well in practice, and why it must be checked under stress.
- **Q₁₂ is positive even with equal voltage magnitudes**: the line consumes reactive power that grows with the transfer (the previous section).

A last rule of thumb that follows from Q₁₂: **reactive power flows from the higher to the lower voltage magnitude**. Active power is tied to angles, reactive power to magnitudes — the "decoupling" that Module 04's fast power flow methods exploit.

## Check your understanding

**Power and power factor**

1. A motor draws 10 kW at power factor 0.8 from 230 V. What are |S|, Q and the current? What happens to the current if the power factor drops to 0.6?
2. Why does a power plant's generator rating appear in MVA, not MW?

**Power transfer**

1. Using the table, how large an angle does the line need to carry 400 MW? What happens to that angle if a parallel line is built (X halves)?
2. In the transfer formula, raise |V₁| by 5 % and keep δ fixed. Which changes more, P₁₂ or Q₁₂? What does that tell you about controlling voltage magnitudes?